# **GPU Memory Monitoring & Management**

---

## How PyTorch Uses GPU Memory

PyTorch does **not** hand each tensor straight to `cudaMalloc`. It uses a **caching allocator**: it grabs large blocks from the driver and reuses them, so freeing a tensor returns memory to PyTorch’s cache, *not* to the GPU. This makes two numbers different and both worth watching:

* **allocated** — bytes currently held by live tensors.
* **reserved** (a.k.a. cached) — bytes the allocator has taken from the driver; always ≥ allocated.

An `OutOfMemoryError` happens when *reserved + a new request* exceeds the device, which is why you can OOM even though `allocated` looks small — the gap is **fragmentation**.

## The Monitoring API

```python
import torch

dev = "cuda:0"
MB = 1024 ** 2

print(f"allocated      {torch.cuda.memory_allocated(dev) / MB:7.1f} MB")
print(f"reserved       {torch.cuda.memory_reserved(dev) / MB:7.1f} MB")
print(f"max allocated  {torch.cuda.max_memory_allocated(dev) / MB:7.1f} MB")
print(f"max reserved   {torch.cuda.max_memory_reserved(dev) / MB:7.1f} MB")

# full human-readable breakdown of the allocator state:
print(torch.cuda.memory_summary(dev))

# reset the peak counters between phases (e.g. before/after an epoch):
torch.cuda.reset_peak_memory_stats(dev)
```

Use `max_memory_allocated` to size your batch: run one step, read the peak, and you know the headroom.

## Memory Snapshots (finding *what* holds memory)

The peak numbers tell you *how much*; a **snapshot** tells you *which call sites* allocated it. Record the allocation history, dump it, and visualize at `https://pytorch.org/memory_viz`.

```python
import torch

# start recording every alloc/free with its Python stack trace
torch.cuda.memory._record_memory_history(max_entries=100_000)

train_a_few_steps()

# dump the timeline to a pickle, then upload it to the memory_viz tool
torch.cuda.memory._dump_snapshot("mem_snapshot.pickle")

# stop recording when done (passing None / enabled=None)
torch.cuda.memory._record_memory_history(enabled=None)
```

The visualizer shows a flame-graph of allocations over time — the easiest way to spot a tensor you forgot to free or a leak that grows each step.

## `empty_cache`, Fragmentation & OOM Debugging

**`torch.cuda.empty_cache()`** returns *reserved-but-unused* blocks to the driver. It does **not** free live tensors and does **not** speed up training — use it only to hand memory back to other processes or to reduce fragmentation, not as a routine fix.

Practical OOM checklist:

```python
# 1) Don't accumulate the autograd graph just to log a number:
total_loss += loss.item()          # .item()/.detach(), NOT  += loss

# 2) Free references before the next iteration if they're large:
del logits, loss
torch.cuda.empty_cache()           # optional; mainly helps fragmentation

# 3) Don't track grads where you don't need them:
with torch.no_grad():
    preds = model(x)
```

For fragmentation-driven OOMs, set the allocator to use expandable segments **before** CUDA initializes:

```bash
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
```

Other levers: smaller batch size, gradient accumulation, mixed precision (AMP), and activation checkpointing — each trades compute or steps for peak memory.

## References

* `torch.cuda` memory management: https://pytorch.org/docs/stable/notes/cuda.html#memory-management
* Memory stats API: https://pytorch.org/docs/stable/cuda.html#memory-management
* Understanding & visualizing snapshots: https://pytorch.org/blog/understanding-gpu-memory-1/
* Memory visualizer: https://pytorch.org/memory_viz
* `PYTORCH_CUDA_ALLOC_CONF`: https://pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf